In [ ]:
import sys, platform, subprocess
import torch

print("===== Environment Info =====")
print("OS:", platform.platform())
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA (torch built with):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU compute capability:", torch.cuda.get_device_capability(0))
    print("GPU total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

try:
    import transformers
    print("transformers:", transformers.__version__)
except ImportError:
    print("transformers: not installed")

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
).stdout

lines = result.splitlines()

# remove the date/time on the first line
if lines:
    lines = lines[1:]

print("\n".join(lines))

===== Environment Info =====
OS: Linux-6.6.122+-x86_64-with-glibc2.39
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA (torch built with): 12.8
cuDNN version: 91900
GPU available: True
GPU name: Tesla T4
GPU compute capability: (7, 5)
GPU total memory (GB): 15.637086208
transformers: 5.16.1
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4            

In [ ]:
import os
os.environ['PATH'] += ':/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64'
!nsys --version

NVIDIA Nsight Systems version 2025.1.1.0


# GPU warm-up

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/whole_inference_estimation/"
!python whole_inference_estimation.py --dtype float32 --reversed_batch false --only_batch 1 --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/1_whole_inference_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 777kB/s]

model.safetensors: downloading bytes:  59% 325M/548M [00:01<00:01, 215MB/s, 28.1MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:02<00:00, 252MB/s, 41.9MB/s  ]
model.safetensors: reconstructing file:  73% 402M/548M [00:02<00:00, 179MB/s, 25.6MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:02<00:00, 181MB/s, 42.3MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:02<00:00, 209MB/s, 49.7MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 4079.31it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 469kB/s]
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_si

# Clock (585 MHz)-fixed execution

In [ ]:
!nvidia-smi -lgc 585
!nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,power.limit --format=csv

GPU clocks set to "(gpuClkMin 585, gpuClkMax 585)" for GPU 00000000:00:04.0

All done.
clocks.current.sm [MHz], clocks.max.sm [MHz], power.limit [W]
585 MHz, 1590 MHz, 70.00 W


In [ ]:
%%bash
mkdir -p output/power
mkdir -p output/ascending
rm -f output/ascending/fineweb_nsight_clocklocked_585.csv

for B in 1 2 4 8 16; do
    nvidia-smi --query-gpu=timestamp,clocks.sm,power.draw,clocks_throttle_reasons.sw_power_cap,clocks_throttle_reasons.sw_thermal_slowdown \
        --format=csv -lms 200 > output/power/clocklock_585_whole_b${B}_log.csv &
    LOGGER_PID=$!

    python whole_inference_estimation.py --dtype float32 --only_batch $B --csv_name fineweb_nsight_clocklocked_585.csv

    kill $LOGGER_PID
done

nvidia-smi -rgc

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=428,032 (max_batch=16, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 5

=== batch_size=1 (iters=256, tokens to process=262,144, normalize /1) ===
  [repeat 1/5]
    Tokens processed:                          262,144
    Elapsed (wall time, incl. data prep + NLL): 37,806.675 ms
    Total Forward-only (sum of 256 calls):    36,251.386 ms
    Pure Forward (kernel, model forward only): 141.607±0.557 ms/call
    Normalized (/1):                          141.607±0.557 ms/call
    Average NLL:                                3.3423
  [repeat 2/5]
    Tokens processed:                          262,144
    Elapsed (wall time, incl. data prep + NLL): 37,760.081 ms
    Total Forward-only (sum of 256 calls):    36,238.253 ms
    Pure Forward (kernel, model forward only): 141.

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5977.32it/s]


In [ ]:
from google.colab import runtime
runtime.unassign()